# Practical 2: Convert Unstructured Log Data into a Structured Dataset
**Course:** BE05000231 - PDS (Semester V, GTU GSET)
**Course Outcome:** CO1

### Objective:
Convert unstructured text logs into a structured Pandas DataFrame via compiled regular expressions, handle edge cases (IPv6, dash bytes, missing fields), log unparseable rejects, and serialize to Apache Parquet.

## 1. Imports and Setup

In [ ]:
import sys
from pathlib import Path
sys.path.append('..')
import pandas as pd

from pipeline.parser import parse_file, parse_report

LOG_PATH = Path('../tests/fixtures/sample_access.log')
REJECTS_PATH = Path('../data/interim/rejects.txt')
PARQUET_PATH = Path('../data/processed/structured_logs.parquet')


## 2. Execute Structured Parsing

In [ ]:
df, rejects = parse_file(
    LOG_PATH,
    chunk_size=1000,
    rejects_path=REJECTS_PATH,
    output_parquet=PARQUET_PATH
)

print(f'Successfully parsed rows: {len(df)}')
print(f'Rejected malformed lines: {len(rejects)}')


## 3. Parse Quality & Audit Report

In [ ]:
report = parse_report(total_attempted=len(df) + len(rejects), parsed_count=len(df), rejected_count=len(rejects))
for k, v in report.items():
    print(f'{k}: {v}')


## 4. Inspection of Structured DataFrame Schema and Types

In [ ]:
print(df.info())
df.head(8)


## 5. Audit Rejected Lines
Attack traffic often contains corrupted or non-standard bytes. Segregating rejects prevents pipeline crashes while retaining evidence.

In [ ]:
print('--- Rejected Malformed Lines ---')
for line_no, raw in rejects:
    print(f'Line {line_no}: {raw}')


## 6. Parquet Verification

In [ ]:
loaded_parquet = pd.read_parquet(PARQUET_PATH)
print(f'Parquet verified. Row count: {len(loaded_parquet)}, Columns: {list(loaded_parquet.columns)}')


### Conclusion:
Practical 2 demonstrated robust parsing of unstructured log data into a typed, memory-efficient Pandas DataFrame and Parquet storage, retaining audit traces of rejected lines.